### Test des tables possibles de pipeline param (test avec class simple)

In [1]:
from enum import Enum
from typing import Optional
from datetime import datetime
import json

In [2]:
# Mappage des paramettre de transformation (mook test de BDD)
class ConditionOperator(Enum):
    GT = ">"     
    LT = "<"     
    GTE = ">="    
    LTE = "<="    
    EQ = "="     
    NEQ = "!="    
    IN = "in"
    NOT_IN = "not_in"



class ETLTransformationCondition:
    def __init__(self,
                 id_condition: int,
                 id_transformation: int,
                 groupe_code: int,
                 operator: ConditionOperator,
                 value_num: float = None,
                 value_str: str = None):
        
        self.id_condition = id_condition
        self.id_transformation = id_transformation
        self.groupe_code = groupe_code  # les conditions sont reliées à une transformation, si elles sont dans le même groupe alors ce sont des AND, sinon c'est d'OR
        self.operator = operator
        self.value_num = value_num
        self.value_str = value_str


class ConditionFailBehavior(Enum):
    SKIP = "skip"       # si la contion passe pas, on passe à la transformation suivante
    STOP = "stop"       # si la contion passe pas, on arrête de transformer
    ERROR = "error"     # si condition passe pas la ligne est une anomalies

# 
class TypeTransformation(Enum):
    MULTIPLY = "multiply"   # multiplication -> fournir valeur float
    DIVIDE = "divide"       # Diviser        -> fournir valeur float
    ADD = "add"             # Additionner    -> fournir valeur float
    SUBTRACT = "subtract"   # Soustraire     -> fournir valeur float
    POWER = "power"         # Puissance      -> fournir valeur float
    ROUND = "round"         # Arrondir       -> fournir int
    FLOOR = "floor"         # arrondir à l'entier inférieur
    CEIL = "ceil"           # arrondir à l'entier supérieur
    CLIP_MIN = "clip_min"   # valeur min pour éviter les abus (si X > 110 -> x = 100)   -> fournir valeur float
    CLIP_MAX = "clip_max"   # valeur max pour éviter les abus (si X > 110 -> x = 100)   -> fournir valeur float

    UPPER = "upper"         # MAJUSCULE
    LOWER = "lower"         # minuscule
    REPLACE = "replace"     # replace - > fournir old/new 
    REGEX_REPLACE = "regex_replace"     # Regex ?
    SUBSTRING = "substring" # -> start/end

    # Pour les type array, mais ca peut etre des arrays json ou des array [xxx, xxx, xx, xxx], ou des liste avec un séparateur comme "xxxxx;xxxxx;xxxxx;xxxxx"
    ARRAY_UNIQUE = "array_unique"   # supprime doublons
    ARRAY_SLICE = "array_slice"     # ex [0, 1, 2]

    # Pour type date ou type datetime, mais comment savoir si c'est une date format anglais ou format fr ??
    ADD_DAYS = "add_days"           # -> fournir int
    ADD_MONTH = "add_month"
    ADD_YEAR = "add_year"   #
    EXTRACT_DAY = "extract_day"   # fournir string dans string 1
    EXTRACT_MONTH = "extract_month" # fournir string dans string 1
    EXTRACT_YEAR = "extract_year"   # fournir string dans string 1
    DEFAULT_DAY = "default_day"     # fournir un entier qui est un numéro de jour
    DEFAULT_MONTH = "default_month" # fournir un entier qui est le numéro du mois
    DEFAULT_YEAR = "default_year"


class ETLColumnTransformation:
    def __init__(self,
                 id_transformation: int,
                 id_etl_column_mapping: int,
                 ordre: int,
                 type_transformation: TypeTransformation,
                 condition_fail_behavior: ConditionFailBehavior,
                 value_num: float = None,
                 value_int: int = None,
                 value_str: str = None,
                 value_int_2: int = None,
                 value_str_2: str = None,
                 conditions: list = None):
        
        self.id_transformation = id_transformation
        self.id_etl_column_mapping = id_etl_column_mapping
        self.ordre = ordre
        self.type_transformation = type_transformation
        self.condition_fail_behavior = condition_fail_behavior
        
        self.value_num = value_num
        self.value_int = value_int
        self.value_str = value_str
        self.value_int_2 = value_int_2
        self.value_str_2 = value_str_2

        self.conditions = conditions or []

In [3]:
class TypeDonnees(Enum):
    INT = "int"
    DECIMAL = "decimal"
    STRING = "string"
    DATE = "date"
    TIMESTAMP = "timestamp"
    BOOLEAN = "boolean"
    ARRAY_DELIMITED_JSON = "array_delimited_json"    # → "item1;item2;item3"
    ARRAY_JSON = "array_json"              # → '["item1","item2"]'

# Classe de base pour les contraintes
class ColumnConstraint:
    def __init__(self, id_constraint: int):
        self.id_constraint = id_constraint

# Contraintes STRING
class StringConstraint(ColumnConstraint):
    def __init__(self, id_constraint: int, min_length: int = 0, max_length: int = None):
        super().__init__(id_constraint)
        self.min_length = min_length
        self.max_length = max_length

# Contraintes NUMERIC
class NumericConstraint(ColumnConstraint):
    def __init__(self, id_constraint: int, nb_min: float = None,
                 nb_max: float = None, nb_decimal: int = None):
        super().__init__(id_constraint)
        self.nb_min = nb_min
        self.nb_max = nb_max
        self.nb_decimal = nb_decimal

# Contraintes DATE
class DateConstraint(ColumnConstraint):
    def __init__(self, id_constraint: int,
                 date_min: datetime = None,
                 date_max: datetime = None):
        super().__init__(id_constraint)
        self.date_min = date_min
        self.date_max = date_max

# ETLColumnMapping (classe principale)
# Ici on compose avec une contrainte (et pas héritage direct)
class ETLColumnMapping:
    def __init__(self,
                 id_etl_column_mapping: int,
                 colonne_bdd: str,
                 colonne_fichier: str,
                 in_file: bool,
                 type_donnees: TypeDonnees,
                 nullable: bool,
                 valeur_defaut: str,
                 unique_constraint: bool,
                 constraint: ColumnConstraint = None,
                 transformations: ETLColumnTransformation = None):
        
        self.id_etl_column_mapping = id_etl_column_mapping
        self.colonne_bdd = colonne_bdd
        self.colonne_fichier = colonne_fichier
        self.in_file = in_file
        self.type_donnees = type_donnees
        self.nullable = nullable
        self.valeur_defaut = valeur_defaut
        self.unique_constraint = unique_constraint
        
        # la contrainte dépend du type
        self.constraint = constraint
        self.transformations = transformations


class ExtensionFichier(Enum):
    CSV = "csv"
    JSON = "json"

class PipelineETL:
    def __init__(self, id_etl_pipeline: int, libelle: str, table_nom: str,
                 dossier_emplacement: str, nom_fichier_fixe: str,
                 nom_fichier_variable: str,
                 extension_fichier: ExtensionFichier,
                 active: bool,
                 colonnes: list[ETLColumnMapping] = None):

        self.id_etl_pipeline = id_etl_pipeline
        self.libelle = libelle
        self.table_nom = table_nom
        self.dossier_emplacement = dossier_emplacement
        self.nom_fichier_fixe = nom_fichier_fixe
        self.nom_fichier_variable = nom_fichier_variable
        self.extension_fichier = extension_fichier
        self.active = active
        self.colonnes = colonnes or []

# MOOK BDD
def get_pipelineBDD_TODO():

    # -------------------------
    # AGE
    # -------------------------
    col_age = ETLColumnMapping(
        id_etl_column_mapping=1,
        colonne_bdd="age",
        colonne_fichier="Age",
        in_file=True,
        type_donnees=TypeDonnees.INT,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=NumericConstraint(1, nb_min=0, nb_max=110, nb_decimal=0),
        transformations=[
            # Si age > 100 → clip à 100
            ETLColumnTransformation(
                id_transformation=1,
                id_etl_column_mapping=1,
                ordre=1,
                type_transformation=TypeTransformation.CLIP_MAX,
                condition_fail_behavior=ConditionFailBehavior.SKIP,
                value_num=100,
                conditions=[
                    ETLTransformationCondition(1, 1, 1, ConditionOperator.GT, value_num=100)
                ]
            ),

            # Si age < 0 → erreur
            # ETLColumnTransformation(
            #     id_transformation=2,
            #     id_etl_column_mapping=1,
            #     ordre=2,
            #     type_transformation=TypeTransformation.ADD, 
            #     condition_fail_behavior=ConditionFailBehavior.ERROR,
            #     value_num=0,
            #     conditions=[
            #         ETLTransformationCondition(2, 2, 1, ConditionOperator.LT, value_num=0)
            #     ]
            # )
        ]
    )

    # -------------------------
    # GENDER
    # -------------------------
    col_gender = ETLColumnMapping(
        id_etl_column_mapping=2,
        colonne_bdd="sexe",
        colonne_fichier="Gender",
        in_file=True,
        type_donnees=TypeDonnees.STRING,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=StringConstraint(1, min_length=2, max_length=20),
        transformations=[
            # Normalisation en majuscule
            ETLColumnTransformation(
                id_transformation=3,
                id_etl_column_mapping=2,
                ordre=1,
                type_transformation=TypeTransformation.UPPER,
                condition_fail_behavior=ConditionFailBehavior.SKIP
            ),

            # Remplacer valeurs incohérentes
            ETLColumnTransformation(
                id_transformation=4,
                id_etl_column_mapping=2,
                ordre=2,
                type_transformation=TypeTransformation.REPLACE,
                condition_fail_behavior=ConditionFailBehavior.SKIP,
                value_str="FEMALE",
                value_str_2="F",
                conditions=[
                    ETLTransformationCondition(3, 4, 1, ConditionOperator.EQ, value_str="FEMALE")
                ]
            )
        ]
    )

    # -------------------------
    # TYPE MALADIE
    # -------------------------
    col_type_maladie = ETLColumnMapping(
        id_etl_column_mapping=3,
        colonne_bdd="type_maladie",
        colonne_fichier="Disease_Type",
        in_file=True,
        type_donnees=TypeDonnees.STRING,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=StringConstraint(1, min_length=2, max_length=50),
        transformations=[
            # Nettoyage string
            ETLColumnTransformation(
                id_transformation=5,
                id_etl_column_mapping=3,
                ordre=1,
                type_transformation=TypeTransformation.LOWER,
                condition_fail_behavior=ConditionFailBehavior.SKIP
            ),

            # Regex replace (ex: enlever caractères spéciaux)
            ETLColumnTransformation(
                id_transformation=6,
                id_etl_column_mapping=3,
                ordre=2,
                type_transformation=TypeTransformation.REGEX_REPLACE,
                condition_fail_behavior=ConditionFailBehavior.SKIP,
                value_str="[^a-zA-Z ]",
                value_str_2=""
            )
        ]
    )

    # -------------------------
    # WEIGHT
    # -------------------------
    col_weight_kg = ETLColumnMapping(
        id_etl_column_mapping=4,
        colonne_bdd="poids_kg",
        colonne_fichier="Weight_kg",
        in_file=True,
        type_donnees=TypeDonnees.DECIMAL,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=NumericConstraint(1, nb_min=0, nb_max=200, nb_decimal=1),
        transformations=[
            # Arrondir à 1 décimale
            ETLColumnTransformation(
                id_transformation=7,
                id_etl_column_mapping=4,
                ordre=1,
                type_transformation=TypeTransformation.ROUND,
                condition_fail_behavior=ConditionFailBehavior.SKIP,
                value_int=1
            ),

            # Si > 200 → ERROR
            ETLColumnTransformation(
                id_transformation=8,
                id_etl_column_mapping=4,
                ordre=2,
                type_transformation=TypeTransformation.CLIP_MAX,
                condition_fail_behavior=ConditionFailBehavior.ERROR,
                value_num=200,
                conditions=[
                    ETLTransformationCondition(4, 8, 1, ConditionOperator.GT, value_num=102)
                ]
            )
        ]
    )

    # -------------------------
    # HEIGHT
    # -------------------------
    col_height_cm = ETLColumnMapping(
        id_etl_column_mapping=5,
        colonne_bdd="taille_cm",
        colonne_fichier="Height_cm",
        in_file=True,
        type_donnees=TypeDonnees.INT,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=NumericConstraint(1, nb_min=0, nb_max=250, nb_decimal=0),
        transformations=[
            # Convertir en mètres (cm -> m)
            ETLColumnTransformation(
                id_transformation=9,
                id_etl_column_mapping=5,
                ordre=1,
                type_transformation=TypeTransformation.DIVIDE,
                condition_fail_behavior=ConditionFailBehavior.SKIP,
                value_num=100
            ),

            # Condition complexe :
            ETLColumnTransformation(
                id_transformation=10,
                id_etl_column_mapping=5,
                ordre=2,
                type_transformation=TypeTransformation.CLIP_MIN,
                condition_fail_behavior=ConditionFailBehavior.ERROR,
                value_num=50
            )
        ]
    )

    # -------------------------
    # PIPELINE
    # -------------------------
    pipeline1 = PipelineETL(
        1,
        "Import diet recommendation",
        "dataset_recommendations_regime",
        "\\raw\\",
        "diet_recommendations_dataset",
        "",
        ExtensionFichier.CSV,
        True,
        [col_age, col_gender, col_type_maladie, col_weight_kg, col_height_cm]
    )

    return pipeline1

# Le but de ce notebook

L’objectif ici est de créer des fonctions réutilisables qui seront exploitées dans des pipelines de traitement de données.

Ces fonctions couvrent plusieurs étapes :

- Ouverture et lecture de fichiers (CSV / JSON)
- Nettoyage et vérification des données
- Séparation des lignes valides des anomalies

Ces fonctions doivent être le plus génériques possible, sans dépendre de valeurs codées en dur, afin d’être facilement réutilisables.

J’expérimente l’idée d’un pipeline "écrit en BDD", en schématisant les données plutôt qu’en utilisant de réelles bases de données, pour tester le concept.

## Partie 1 : Ouverture de fichiers

Les fonctions de cette partie servent à ouvrir un ou plusieurs fichiers CSV ou JSON dans le cadre d’une pipeline. Elles permettent :

- De localiser les fichiers dans les dossiers
- De détecter leur type (CSV / JSON)
- De les convertir en DataFrame avec pandas, pour les exploiter ensuite sans se soucier des extensions

In [4]:
import os
import re
import pandas as pd
import numpy as np

DATA_ROOT = "../data"

### 1.1 Normalisation du chemin

Cette fonction permet de déterminer le dossier où se trouvent les fichiers à traiter.

- Elle se base sur une variable d’environnement comme DATA_ROOT
- On peut ensuite fournir en argument des sous-dossiers où l’on imagine stocker les fichiers CSV ou JSON

Cela garantit que les pipelines restent flexibles et portables.

In [5]:
# 1. Normalisation du chemin
def normalize_path(dossier_emplacement, data_root=DATA_ROOT):
    """
    Normalise le chemin du dossier relatif à ./data/
    """
    # Normaliser slashes
    dossier_emplacement = dossier_emplacement.replace("\\", "/").replace("//", "/")
    # Supprimer slash en début ou fin
    dossier_emplacement = dossier_emplacement.strip("/")
    # Construire chemin complet
    full_path = os.path.join(data_root, dossier_emplacement)
    # Normaliser le chemin final
    return os.path.normpath(full_path)

### 1.2 Création du motif Regex pour trouver le fichier

Cette fonction génère un motif regex pour identifier les fichiers à traiter.

- Arguments : nom_fixe, nom_variable, extension
    - nom_fixe : première partie du nom du fichier (ex. "Exercice")
    - nom_variable : partie variable du nom du fichier (ex. "YYYYMMDD")
    - extension : type de fichier (csv ou json)

Le motif généré permet de retrouver tous les fichiers correspondants, par exemple :

- CSV commençant par "Exercice"
- Avec 8 caractères pour la date
- Se terminant par .csv

In [6]:
# 2. Création du motif regex pour le fichier
def build_filename_pattern(nom_fixe, nom_variable, extension):
    if not extension.startswith("."):
        extension = "." + extension
    
    if nom_variable:
        # Le nom variable a une longueur connue : on remplace par autant de "."
        motif = f"^{re.escape(nom_fixe)}{'.' * len(nom_variable)}{re.escape(extension)}$"
    else:
        motif = f"^{re.escape(nom_fixe)}{re.escape(extension)}$"
    
    return motif

### 1.3 Recherche des fichiers correspondants

Cette fonction cherche les fichiers dans un dossier en fonction de la regex fournie :

- Arguments : dossier et motif regex
- Retour : liste des chemins de fichiers correspondants

⚠️ Actuellement, un print est utilisé si le dossier n’existe pas.

- En production, il serait préférable de remonter une erreur ou de logger l’information.

In [7]:
# 3. Recherche des fichiers correspondants et retourne les path
def find_matching_files(dossier, motif_regex):
    matched_files = []
    if os.path.exists(dossier) and os.path.isdir(dossier):
        for f in os.listdir(dossier):
            if re.match(motif_regex, f):
                matched_files.append(os.path.join(dossier, f))  # chemin complet
    else:
        print(f"[WARNING] Le dossier '{dossier}' n'existe pas.")
    return matched_files


# --- TESTS ---
dossier_emplacement = "\\raw\\"
nom_fichier_fixe = "exercisedb_hobby"
nom_fichier_variable = ""  # par exemple une date
extension_fichier = ".json"

# Normalisation du chemin
normalized_folder = normalize_path(dossier_emplacement)
print("Chemin normalisé :", normalized_folder)

# Construction du motif regex
pattern = build_filename_pattern(nom_fichier_fixe, nom_fichier_variable, extension_fichier)
print("Motif regex :", pattern)

# Recherche des fichiers
matched = find_matching_files(normalized_folder, pattern)
print("Fichiers trouvés :", matched)


# --- Autre exemple ---
dossier_emplacement2 = "raw/"
nom_fichier_fixe2 = "exercisedb_hobby"
nom_fichier_variable2 = ""
extension_fichier2 = "json"

normalized_folder2 = normalize_path(dossier_emplacement2)
pattern2 = build_filename_pattern(nom_fichier_fixe2, nom_fichier_variable2, extension_fichier2)
matched2 = find_matching_files(normalized_folder2, pattern2)

print("\nChemin normalisé 2 :", normalized_folder2)
print("Motif regex 2 :", pattern2)
print("Fichiers trouvés 2 :", matched2)

Chemin normalisé : ..\data\raw
Motif regex : ^exercisedb_hobby\.json$
Fichiers trouvés : ['..\\data\\raw\\exercisedb_hobby.json']

Chemin normalisé 2 : ..\data\raw
Motif regex 2 : ^exercisedb_hobby\.json$
Fichiers trouvés 2 : ['..\\data\\raw\\exercisedb_hobby.json']


### 1.4 Ouverture et lecture des fichiers

Cette fonction prend un **chemin de fichier unique** et :

- Tente de le lire avec pandas selon son type (CSV ou JSON)
- Retourne un **DataFrame unique**
- Pour les CSV :
  - On considère que chaque fichier a une ligne header
  - Les lignes qui ne correspondent pas au nombre de colonnes du header sont automatiquement ignorées

⚠️ Actuellement, un print s’affiche si :

- Ce n’est ni un CSV ni un JSON
- Le fichier n’existe pas ou ne peut pas être lu
- En production, ces prints devraient être remplacés par des logs ou exceptions

💡 Notes complémentaires :

- Une seconde fonction `read_files` permet de traiter une **liste de fichiers**, mais elle utilise simplement `read_single_file` en boucle. Elle est moins utile si l’on souhaite suivre individuellement chaque fichier (logs, comptage des lignes ignorées, etc.)
- On pourrait également comptabiliser le nombre de lignes supprimées dans les CSV pour un suivi et un logging plus précis

In [8]:
# Lecture automatique avec pandas
def read_single_file_with_pandas(file_path):
    """
    Lit un fichier CSV ou JSON et retourne un DataFrame.
    Retourne None si l'extension n'est pas supportée ou en cas d'erreur.
    """
    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == ".csv":
            df = pd.read_csv(file_path, header=0, on_bad_lines='skip') # header obligatoire et saute les lignes incorrectes (nb colonne different du header)
        elif ext == ".json":
            df = pd.read_json(file_path)
        else:
            print(f"[WARNING] Extension non supportée pour {file_path}, ignoré.")
            return None
        return df
    except Exception as e:
        print(f"[ERROR] Impossible de lire {file_path}: {e}")
        return None


def read_files_with_pandas(file_paths):
    """
    Lit une liste de fichiers CSV ou JSON et retourne une liste de DataFrames
    en utilisant read_single_file_with_pandas.
    """
    dfs = []
    for fpath in file_paths:
        df = read_single_file_with_pandas(fpath)
        if df is not None:
            dfs.append(df)
    return dfs


# ---TEST---
dfs = read_files_with_pandas(matched)
print(f"Nombre de DataFrames lus : {len(dfs)}")
if dfs:
    print(dfs[0].head())  # afficher les 5 premières lignes du premier fichier

Nombre de DataFrames lus : 1
             exerciseId                                   name  \
0  exr_41n2ha5iPFpN3hEJ                                   es     
1  exr_41n2haAabPyN5t8y                                     10   
2  exr_41n2hadPLLFRGvFk        Sliding Floor Pulldown on Towel   
3  exr_41n2hadQgEEX8wDN                            Triceps Dip   
4  exr_41n2haNJ3NA8yCE2  Dumbbell Incline One Arm Hammer Press   

                                            imageUrl              bodyParts  \
0                                               True                [WAIST]   
1  https://cdn.exercisedb.dev/media/w/images/RB7N...   [QUADRICEPS, THIGHS]   
2  https://cdn.exercisedb.dev/media/w/images/9fW9...                 [BACK]   
3  https://cdn.exercisedb.dev/media/w/images/MruS...  [TRICEPS, UPPER ARMS]   
4  https://cdn.exercisedb.dev/media/w/images/qkXo...           [UPPER ARMS]   

                                          equipments exerciseType  \
0                             

### 1.5 Ptt équivalent au downloader ?

In [9]:
def get_df_matched_files(pipeline: PipelineETL):
    """
    À partir d'une instance PipelineETL, retourne la liste des fichiers/dataframe correspondants.
    """
    # 1. Normalisation du chemin
    normalized_folder = normalize_path(pipeline.dossier_emplacement)
    # print("[debug] get_df_matched_files - normalized_folder : " + normalized_folder)

    # 2. Construction du motif regex
    pattern = build_filename_pattern(
        pipeline.nom_fichier_fixe,
        pipeline.nom_fichier_variable,
        pipeline.extension_fichier.value  # Enum → string
    )
    # print("[debug] get_df_matched_files - pattern : " + pattern)

    # 3. Recherche des fichiers
    matched_files_path = find_matching_files(normalized_folder, pattern)
    # print("[debug] get_df_matched_files - Fichiers trouvés :", matched_files_path)

    # 4. Ouvre les fichier en retournant une liste de dataframe
    df_matched = read_files_with_pandas(matched_files_path)

    return df_matched

## Partie 2 : Nettoyage et vérification des données

### 2.1 ColumnMapper 

Responsabilité :

- renommer colonnes
- créer colonnes manquantes

Avec en argument un dataframe et une liste de la class [ETLColumnMapping]

In [10]:
def column_mapper(
    df: pd.DataFrame,
    mappings: list[ETLColumnMapping]
) -> pd.DataFrame:
    result = pd.DataFrame()

    for m in mappings:
        if m.in_file and m.colonne_fichier in df.columns:
            result[m.colonne_bdd] = df[m.colonne_fichier]
        else:
            result[m.colonne_bdd] = None

    return result

### 2.2 Créer un dataframe pour mettre les anomalies

In [11]:
def generate_anomaly_dataframe(mappings: list[ETLColumnMapping]) -> pd.DataFrame:
    columns = [m.colonne_bdd for m in mappings]
    columns += ["erreur"]
    
    result = pd.DataFrame(columns=columns)
    
    return result

### 2.3 Nettoyer les colonnes texte d'un DataFrame (supprime les espaces inutiles)

In [12]:
def clean_txt(df):
    """
    Nettoie les colonnes texte d'un DataFrame :
    - Supprime les espaces en début et fin de chaque cellule (pour les strings uniquement).
    - Remplace les cellules vides ou ne contenant que des espaces par NaN.
    - Ne traite que les colonnes contenant des strings, ignore les autres types (listes, dicts, etc.).
    """
    # On travaille sur une copie pour ne pas modifier l'original
    df_clean = df.copy()
    
    # Sélectionner les colonnes texte (type object)
    object_columns = df_clean.select_dtypes(include='object').columns
    
    for col in object_columns:
        # Appliquer le nettoyage seulement aux valeurs qui sont des strings
        def clean_cell(value):
            if isinstance(value, str):
                # Supprimer les espaces en début/fin
                cleaned = value.strip()
                # Remplacer par NaN si vide ou que des espaces
                return np.nan if cleaned == "" else cleaned
            else:
                # Laisser les non-strings tranquilles (listes, dicts, etc.)
                return value
        
        df_clean[col] = df_clean[col].apply(clean_cell)
    
    return df_clean

### 2.4 Vérifier les valeurs NULL

In [13]:
def handle_missing_values(
    df: pd.DataFrame,
    anomaly_df: pd.DataFrame,
    mappings: list[ETLColumnMapping]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Traite les valeurs manquantes dans df selon les règles de mappings :
    - Convertit les valeurs vides / null / NaN en pd.NA
    - Si nullable=True : laisse pd.NA
    - Si nullable=False et valeur_defaut définie : remplace les NA par la valeur par défaut
    - Sinon : les lignes avec NA deviennent anomalies
    """
    df_clean = df.copy()
    anomalies = anomaly_df.copy()

    for mapping in mappings:
        col = mapping.colonne_bdd
        if col not in df_clean.columns:
            continue

        # --- Étape 1 : standardiser les valeurs "vides" en pd.NA ---
        df_clean[col] = df_clean[col].replace(
            ["", " ", None, "null", "NULL"], pd.NA
        )

        # --- Étape 2 : vérifier nullable ---
        mask_na = df_clean[col].isna()
        if not mask_na.any():
            # pas de valeur manquante → passer à la colonne suivante
            continue

        if mapping.nullable:
            # nullable = True → laisser pd.NA
            continue

        if mapping.valeur_defaut not in ["", " ", None, "null", "NULL", np.nan]:
            # nullable = False mais valeur par défaut définie → remplacer les NA
            df_clean[col] = df_clean[col].fillna(mapping.valeur_defaut)
        else:
            # nullable = False et pas de valeur par défaut → ce sont des anomalies
            # capturer les lignes en anomalies avant suppression
            rows_with_error = df_clean.loc[mask_na].copy()
            rows_with_error["erreur"] = f"La cellule {col} est null"
            anomalies = pd.concat([anomalies, rows_with_error], ignore_index=True)

            # Supprimer ces lignes du df_clean
            df_clean = df_clean.loc[~mask_na].copy()

    df_clean.reset_index(drop=True, inplace=True)
    anomalies.reset_index(drop=True, inplace=True)

    return df_clean, anomalies

### 2.5 Regarder si les colonnes sont convertibles dans le type cible

In [14]:
def convert_column_type(
    df: pd.DataFrame,
    anomaly_df: pd.DataFrame,
    mappings: list[ETLColumnMapping]
) -> tuple[pd.DataFrame, pd.DataFrame]:

    df_clean = df.copy()
    anomalies = anomaly_df.copy()

    for mapping in mappings:
        col = mapping.colonne_bdd

        if col not in df_clean.columns:
            continue

        series = df_clean[col]

        #region --- Étape 1 : conversion selon type de toutes les lignes de la colonne ---
        if mapping.type_donnees == TypeDonnees.INT:
            # Conversion simple: numérique -> arrondi 0 décimale -> entier nullable
            converted = pd.to_numeric(series, errors='coerce').round(0).astype('Int64')
        elif mapping.type_donnees == TypeDonnees.DECIMAL:
            converted = pd.to_numeric(series, errors='coerce')
        elif mapping.type_donnees == TypeDonnees.STRING:
            converted = series.astype(str)
        elif mapping.type_donnees == TypeDonnees.ARRAY_DELIMITED_JSON:
            # Convertir les listes en string délimité par ";"
            separator = ";"
            converted = series.apply(
                lambda x: separator.join(map(str, x)) if isinstance(x, list) else str(x)
            )
        elif mapping.type_donnees == TypeDonnees.ARRAY_JSON:
            converted = series.apply(
                lambda x: json.dumps(x) if isinstance(x, list) else json.dumps([x])
            )
        elif mapping.type_donnees in [TypeDonnees.DATE, TypeDonnees.TIMESTAMP]:
            converted = pd.to_datetime(series, errors='coerce')
            if mapping.type_donnees == TypeDonnees.DATE:
                converted = converted.dt.date
        elif mapping.type_donnees == TypeDonnees.BOOLEAN:
            converted = series.map(lambda x: True if str(x).lower() in ['true','1']
                                   else False if str(x).lower() in ['false','0'] else pd.NA)
        else:
            converted = series
        #endregion

        #region --- Étape 2 : gérer les valeurs manquantes selon nullable / valeur par défaut ---
        mask_invalid = converted.isna()

        # Si valeur null ok alors ne rien faire de plus
        if mapping.nullable:
            # On laisse les NaN
            df_clean[col] = converted
            continue

        if mapping.valeur_defaut not in [None, "", np.nan]:
            # Convertir valeur par défaut dans le bon type
            try:
                if mapping.type_donnees == TypeDonnees.INT:
                    # Même logique pour le défaut: numérique -> arrondi -> int
                    default_value = int(round(float(mapping.valeur_defaut)))
                elif mapping.type_donnees == TypeDonnees.DECIMAL:
                    default_value = pd.to_numeric(mapping.valeur_defaut)
                elif mapping.type_donnees in [TypeDonnees.STRING, TypeDonnees.ARRAY_DELIMITED_JSON, TypeDonnees.ARRAY_JSON]:
                    default_value = str(mapping.valeur_defaut)
                elif mapping.type_donnees in [TypeDonnees.DATE, TypeDonnees.TIMESTAMP]:
                    default_value = pd.to_datetime(mapping.valeur_defaut)
                    if mapping.type_donnees == TypeDonnees.DATE:
                        default_value = default_value.date()
                elif mapping.type_donnees == TypeDonnees.BOOLEAN:
                    default_value = True if str(mapping.valeur_defaut).lower() in ['true','1'] else False
                else:
                    default_value = mapping.valeur_defaut
            except Exception:
                # Mettre à jour les valeurs valides
                df_clean.loc[~mask_invalid, col] = converted.loc[~mask_invalid]
                # Valeur par défaut invalide : toutes les lignes invalides deviennent anomalies
                rows_with_error = df_clean.loc[mask_invalid].copy()
                rows_with_error["erreur"] = f"La cellule {col} n'est pas convertible et la valeur par défaut '{mapping.valeur_defaut}' est invalide"
                anomalies = pd.concat([anomalies, rows_with_error], ignore_index=True)
                # On supprime ces lignes
                df_clean = df_clean.loc[~mask_invalid]
                continue

            # Remplacer les NaN par valeur par défaut
            converted[mask_invalid] = default_value
            df_clean[col] = converted
        else:
            # Ni nullable, ni valeur par défaut : ce sont des anomalies
            rows_with_error = df_clean.loc[mask_invalid].copy()
            rows_with_error["erreur"] = f"La cellule {col} n'est pas convertible en {mapping.type_donnees.value}"
            anomalies = pd.concat([anomalies, rows_with_error], ignore_index=True)
            # Supprimer ces lignes du df_clean
            df_clean = df_clean.loc[~mask_invalid]
        #endregion

    df_clean.reset_index(drop=True, inplace=True)
    anomalies.reset_index(drop=True, inplace=True)

    return df_clean, anomalies


### 2.6 Transformer les données

##### Évalue une condition simple sur une colonne, retourne un masque booléen

In [15]:
def evaluate_single_condition(df: pd.DataFrame, col: str, condition: ETLTransformationCondition) -> pd.Series:
    """
    Évalue une condition simple sur une colonne, retourne un masque booléen.
    """
    series = df[col]

    # Compare chaque ligne selon l'opérateur de la condition
    if condition.operator == ConditionOperator.GT:
        return series > condition.value_num

    elif condition.operator == ConditionOperator.LT:
        return series < condition.value_num

    elif condition.operator == ConditionOperator.GTE:
        return series >= condition.value_num

    elif condition.operator == ConditionOperator.LTE:
        return series <= condition.value_num

    elif condition.operator == ConditionOperator.EQ:
        return series == (condition.value_str if condition.value_str is not None else condition.value_num)

    elif condition.operator == ConditionOperator.NEQ:
        return series != (condition.value_str if condition.value_str is not None else condition.value_num)

    elif condition.operator == ConditionOperator.IN:
        return series.isin(condition.value_str)

    elif condition.operator == ConditionOperator.NOT_IN:
        return ~series.isin(condition.value_str)

    # Par défaut : aucune ligne ne passe
    return pd.Series(False, index=df.index)


##### Évalue un ensemble de conditions pour une colonne

In [16]:
def evaluate_conditions(df: pd.DataFrame, col: str, conditions: list[ETLTransformationCondition]) -> pd.Series:
    """
    Évalue un ensemble de conditions pour une colonne.
    Retourne un masque booléen indiquant quelles lignes passent les conditions.
    """
    if not conditions:
        return pd.Series(True, index=df.index)

    # On regroupe les conditions par groupe_code pour appliquer une logique AND à l'intérieur du groupe
    groups = {}
    for cond in conditions:
        groups.setdefault(cond.groupe_code, []).append(cond)

    group_results = []

    for group in groups.values():
        group_mask = pd.Series(True, index=df.index)

        for condition in group:
            current_mask = evaluate_single_condition(df, col, condition)
            group_mask &= current_mask # AND entre conditions du groupe

        group_results.append(group_mask)

    # OR entre les groupes pour respecter la logique "au moins un groupe est vrai"
    final_mask = group_results[0]
    for gm in group_results[1:]:
        final_mask |= gm

    return final_mask


##### Ajoute les lignes ayant échoué ou créant anomalies à la table anomalies

In [17]:
def add_to_anomalies(df, df_original, anomalies, mask, message):
    """
    Ajoute les lignes ayant échoué ou créant anomalies à la table anomalies.
    """
    if not mask.any():
        return anomalies

    # Copie des lignes originales pour garder les valeurs initiales
    rows = df_original.loc[mask].copy()
    rows["erreur"] = message

    return pd.concat([anomalies, rows], ignore_index=True)


##### Gestion des lignes qui ne passent pas les conditions (SKIP, STOP, ERROR)

In [18]:
def handle_condition_failure(
    df,
    df_original,
    anomalies,
    transformation,
    col,
    active_mask,
    error_mask_global,
    fail_mask
):
    """
    Gestion des lignes qui ne passent pas les conditions.
    SKIP : ignore les lignes
    STOP : retire les lignes de l'application future
    ERROR : ajoute aux anomalies
    """
    if not fail_mask.any():
        return active_mask, error_mask_global, anomalies

    if transformation.condition_fail_behavior == ConditionFailBehavior.SKIP:
        # On ne fait rien, la transformation sera appliquée seulement aux lignes valides par le parent
        return active_mask, error_mask_global, anomalies

    elif transformation.condition_fail_behavior == ConditionFailBehavior.STOP:
        # Les lignes qui échouent sont retirées du flux actif (stop la transformation pour cette colonne)
        active_mask = active_mask & ~fail_mask
        return active_mask, error_mask_global, anomalies

    elif transformation.condition_fail_behavior == ConditionFailBehavior.ERROR:
        # Les lignes en échec deviennent anomalies
        anomalies = add_to_anomalies(
            df,
            df_original,
            anomalies,
            fail_mask,
            f"Condition non respectée col {col} transfo {transformation.ordre}"
        )

        active_mask = active_mask & ~fail_mask
        error_mask_global = error_mask_global | fail_mask

        return active_mask, error_mask_global, anomalies

    return active_mask, error_mask_global, anomalies


##### Applique la transformation définie sur les lignes sélectionnées

In [19]:
###### TODO : faut utiliser un truc genre (series, errors='coerce') pour etre sur que ca deviennent NaN ICI risque d'erreur cachées ou de crash une colonnes entiere
def apply_transformation(df, col, transformation, mask):
    """
    Applique la transformation définie sur les lignes sélectionnées.
    Chaque type de transformation (MULTIPLY, DIVIDE, ADD, etc.) est géré avec pandas.
    """
    series = df[col]

    try:
        # Préparer version safe numérique (remplace non numérique par NaN), permet de faire les transfo numérique en sécurité.
        safe_num = pd.to_numeric(series[mask], errors="coerce")

        if transformation.type_transformation == TypeTransformation.MULTIPLY:
            df.loc[mask, col] = safe_num * transformation.value_num

        elif transformation.type_transformation == TypeTransformation.DIVIDE:
            df.loc[mask, col] = safe_num / transformation.value_num

        elif transformation.type_transformation == TypeTransformation.ADD:
            df.loc[mask, col] = safe_num + transformation.value_num

        elif transformation.type_transformation == TypeTransformation.SUBTRACT:
            df.loc[mask, col] = safe_num - transformation.value_num

        elif transformation.type_transformation == TypeTransformation.ROUND:
            df.loc[mask, col] = safe_num.round(transformation.value_int)

        elif transformation.type_transformation == TypeTransformation.CLIP_MAX:
            df.loc[mask, col] = safe_num.clip(upper=transformation.value_num)

        elif transformation.type_transformation == TypeTransformation.CLIP_MIN:
            df.loc[mask, col] = safe_num.clip(lower=transformation.value_num)

        # Modif String
        elif transformation.type_transformation == TypeTransformation.UPPER:
            df.loc[mask, col] = series[mask].str.upper()

        elif transformation.type_transformation == TypeTransformation.LOWER:
            df.loc[mask, col] = series[mask].str.lower()

        elif transformation.type_transformation == TypeTransformation.REPLACE:
            df.loc[mask, col] = series[mask].replace(
                transformation.value_str,
                transformation.value_str_2
            )

        elif transformation.type_transformation == TypeTransformation.REGEX_REPLACE:
            df.loc[mask, col] = series[mask].str.replace(
                transformation.value_str,
                transformation.value_str_2,
                regex=True
            )

        elif transformation.type_transformation == TypeTransformation.SUBSTRING:
            df.loc[mask, col] = series[mask].str.slice(
                transformation.value_int,
                transformation.value_int_2
            )

    except Exception:
        df.loc[mask, col] = np.nan
        pass

    return df


##### Détecte si la transformation a créé de nouvelles valeurs NaN

In [20]:
def detect_transformation_errors(df, previous_values, col, success_mask):
    """
    Détecte si la transformation a créé de nouvelles valeurs NaN
    """
    current_values = df[col]

    # Une erreur est détectée si une valeur est devenue NaN alors qu'elle n'était pas NaN avant
    new_nan_mask = (
        success_mask
        & current_values.isna()
        & ~previous_values.isna()
    )

    return new_nan_mask



##### Fonction principale pour appliquer des transformations sur un DataFrame

In [21]:
def apply_transformations(df, anomalies, mappings):
    """
    Fonction principale pour appliquer des transformations sur un DataFrame.
    df : pd.DataFrame - dataframe à nettoyer
    anomalies : pd.DataFrame - dataframe qui contient les anomalies détectées
    mappings : liste - liste d'objets mapping décrivant les colonnes et transformations à appliquer

    Retour :
    df_clean : pd.DataFrame après application des transformations
    anomalies : pd.DataFrame mis à jour avec toutes les anomalies détectées
    """

    # Crée une copie du dataframe pour éviter de modifier l'original directement
    df_clean = df.copy()
    df_original = df.copy()  # garde une version intacte pour enregistrer les anomalies

    # Masque global des erreurs pour savoir quelles lignes sont déjà invalides
    error_mask_global = pd.Series(False, index=df_clean.index)

    # Boucle sur les colonnes
    for mapping in mappings:
        col = mapping.colonne_bdd

        if col not in df_clean.columns:
            continue

        if not mapping.transformations:
            continue

        # Masque des lignes actives : celles qui ne sont pas NaN et qui ne sont pas des anomalies
        active_mask = ~df_clean[col].isna() & ~error_mask_global

        # Tri des transformations par ordre défini dans la configuration
        transformations_sorted = sorted(mapping.transformations, key=lambda t: t.ordre)

        # Boucle sur chaque transformation triée
        for transformation in transformations_sorted:

            working_mask = active_mask # lignes encore traitables

            if not working_mask.any():
                break

            # Évaluation des conditions de la transformation sur les lignes actives
            condition_mask = evaluate_conditions(df_clean, col, transformation.conditions)
            condition_mask = condition_mask & working_mask

            fail_mask = working_mask & ~condition_mask

            # Gestion de l'échec de condition selon la configuration (SKIP, STOP, ERROR)
            active_mask, error_mask_global, anomalies = handle_condition_failure(
                df_clean,
                df_original,
                anomalies,
                transformation,
                col,
                active_mask,
                error_mask_global,
                fail_mask
            )

            # Recalcul du working_mask après gestion des échecs
            working_mask = active_mask

            if not working_mask.any():
                break

            # Masque des lignes qui passent la condition et seront transformées
            success_mask = working_mask & condition_mask

            # Sauvegarde des valeurs avant transformation pour détecter d'éventuelles erreurs
            previous_values = df_clean[col].copy()

            # Application de la transformation sur les lignes valides
            df_clean = apply_transformation(df_clean, col, transformation, success_mask)

            # Détection des erreurs de transformation (valeurs devenues NaN alors qu'elles ne l'étaient pas)
            new_error_mask = detect_transformation_errors(
                df_clean,
                previous_values,
                col,
                success_mask
            )

            # Si des erreurs sont détectées, on les ajoute aux anomalies
            if new_error_mask.any():
                anomalies = add_to_anomalies(
                    df_clean,
                    df_original,
                    anomalies,
                    new_error_mask,
                    f"Erreur transformation col {col} transfo {transformation.ordre}"
                )

                # Retire les lignes en erreur du flux actif
                active_mask = active_mask & ~new_error_mask
                # Ajoute les nouvelles erreurs au masque global
                error_mask_global = error_mask_global | new_error_mask

    # On supprime les lignes complètement invalides (déjà en anomalie)
    df_clean = df_clean[~error_mask_global].copy()

    return df_clean, anomalies



### 2.7 Exécuter les vérifications en fonction des contraintes des colonnes

In [22]:
def _build_numeric_constraint_mask(
    series: pd.Series,
    constraint: NumericConstraint,
    index: pd.Index
) -> pd.Series:
    mask_invalid = pd.Series(False, index=index)
    numeric_series = pd.to_numeric(series, errors='coerce')

    if constraint.nb_min is not None:
        mask_invalid = mask_invalid | (numeric_series < constraint.nb_min)

    if constraint.nb_max is not None:
        mask_invalid = mask_invalid | (numeric_series > constraint.nb_max)

    if constraint.nb_decimal is not None:
        # Vérifie le nombre de décimales en analysant la valeur d'origine en string
        def too_many_decimals(value):
            if pd.isna(value):
                return False
            s = str(value).strip()
            if s == "" or s.lower() in ["nan", "none", "null"]:
                return False
            if "." in s:
                return len(s.split(".")[-1]) > constraint.nb_decimal
            return False

        decimal_mask = series.apply(too_many_decimals)
        mask_invalid = mask_invalid | decimal_mask

    return mask_invalid


def _build_string_constraint_mask(
    series: pd.Series,
    constraint: StringConstraint,
    index: pd.Index
) -> pd.Series:
    mask_invalid = pd.Series(False, index=index)
    str_len = series.astype(str).str.len()

    if constraint.min_length is not None:
        mask_invalid = mask_invalid | (str_len < constraint.min_length)

    if constraint.max_length is not None:
        mask_invalid = mask_invalid | (str_len > constraint.max_length)

    return mask_invalid


def _build_date_constraint_mask(
    series: pd.Series,
    constraint: DateConstraint,
    index: pd.Index
) -> pd.Series:
    mask_invalid = pd.Series(False, index=index)
    dt_series = pd.to_datetime(series, errors='coerce')

    if constraint.date_min is not None:
        mask_invalid = mask_invalid | (dt_series < pd.to_datetime(constraint.date_min))

    if constraint.date_max is not None:
        mask_invalid = mask_invalid | (dt_series > pd.to_datetime(constraint.date_max))

    return mask_invalid


def check_column_constraint(
    df: pd.DataFrame,
    anomaly_df: pd.DataFrame,
    mappings: list[ETLColumnMapping]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Vérifie les contraintes de colonnes selon le type de contrainte défini dans mapping.constraint.
    Les lignes en erreur sont déplacées dans anomalies avec un message explicite.
    """

    df_clean = df.copy()
    anomalies = anomaly_df.copy()

    for mapping in mappings:
        col = mapping.colonne_bdd
        if col not in df_clean.columns:
            continue

        constraint = mapping.constraint
        if constraint is None:
            continue

        # Série sur le DataFrame courant (qui peut changer après suppression de lignes)
        series = df_clean[col]

        # --- Contraintes NUMERIC (INT/DECIMAL) ---
        if isinstance(constraint, NumericConstraint):
            mask_invalid = _build_numeric_constraint_mask(series, constraint, df_clean.index)

        # --- Contraintes STRING / ARRAY stringifiées ---
        elif isinstance(constraint, StringConstraint):
            mask_invalid = _build_string_constraint_mask(series, constraint, df_clean.index)

        # --- Contraintes DATE/TIMESTAMP ---
        elif isinstance(constraint, DateConstraint):
            mask_invalid = _build_date_constraint_mask(series, constraint, df_clean.index)

        # BOOLEAN : pas de contrainte dédiée
        else:
            continue

        if mask_invalid.any():
            rows_with_error = df_clean.loc[mask_invalid].copy()
            rows_with_error["erreur"] = f"La cellule {col} ne respecte pas les contraintes de la colonne"
            anomalies = pd.concat([anomalies, rows_with_error], ignore_index=True)

            # Supprimer les lignes invalides
            df_clean = df_clean.loc[~mask_invalid].copy()

    df_clean.reset_index(drop=True, inplace=True)
    anomalies.reset_index(drop=True, inplace=True)

    return df_clean, anomalies

### Test

In [23]:
from IPython.display import display # pour notebook test

pipeline_rules = get_pipelineBDD_TODO()
pipeline_column_mapping = pipeline_rules.colonnes

dfs_matched_files = get_df_matched_files(pipeline_rules)

for df in dfs_matched_files:
    df_clean = column_mapper(df, pipeline_column_mapping)
    anomalies = generate_anomaly_dataframe(pipeline_column_mapping)

    # supprime les espaces inutiles des string
    df_clean = clean_txt(df_clean)

    # Verifier les valeurs null
    df_clean, anomalies = handle_missing_values(df_clean, anomalies, pipeline_column_mapping)

    # Regarder si les colonnes sont convertible dans le type cible
    df_clean, anomalies = convert_column_type(df_clean, anomalies, pipeline_column_mapping)

    print("--------------- Avant Transfo -----------")
    display(df_clean)
    # print("----- anomalies FIN -----")
    display(anomalies)

    # Transformer les colonnes
    df_clean, anomalies = apply_transformations(df_clean, anomalies, pipeline_column_mapping)

    print("--------------- Apres Transfo -----------")
    display(df_clean)
    # print("----- anomalies FIN -----")
    display(anomalies)

    # Vérifier les contraintes métier de colonnes
    df_clean, anomalies = check_column_constraint(df_clean, anomalies, pipeline_column_mapping)

    print("--------------- Check Fin -----------")
    display(df_clean)
    # print("----- anomalies FIN -----")
    display(anomalies)

--------------- Avant Transfo -----------


C:\Users\33631\AppData\Local\Temp\ipykernel_1492\4046466031.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df_clean.select_dtypes(include='object').columns


,age,sexe,type_maladie,poids_kg,taille_cm
0,56,Male,Obesity,58.4,160
1,69,Male,Diabetes,101.2,169
2,46,Female,Hypertension,63.5,173
3,60,Male,Diabetes,79.5,197
4,25,Female,Obesity,105.7,156
...,...,...,...,...,...
791,18,Male,Obesity,72.1,160
792,35,Female,Hypertension,104.0,171
793,49,Female,Obesity,56.0,182
794,64,Male,Diabetes,66.6,185


,age,sexe,type_maladie,poids_kg,taille_cm,erreur
0,32,Male,NaN,58.1,164,La cellule type_maladie est null
1,78,Male,NaN,102.2,170,La cellule type_maladie est null
2,41,Male,NaN,83.7,154,La cellule type_maladie est null
3,39,Female,NaN,106.3,198,La cellule type_maladie est null
4,70,Male,NaN,87.7,168,La cellule type_maladie est null
...,...,...,...,...,...,...
199,67,Male,NaN,53.9,153,La cellule type_maladie est null
200,30,Female,NaN,59.2,190,La cellule type_maladie est null
201,63,Female,NaN,77.9,184,La cellule type_maladie est null
202,78,Female,NaN,83.0,197,La cellule type_maladie est null


--------------- Apres Transfo -----------


,age,sexe,type_maladie,poids_kg,taille_cm


,age,sexe,type_maladie,poids_kg,taille_cm,erreur
0,32,Male,NaN,58.1,164,La cellule type_maladie est null
1,78,Male,NaN,102.2,170,La cellule type_maladie est null
2,41,Male,NaN,83.7,154,La cellule type_maladie est null
3,39,Female,NaN,106.3,198,La cellule type_maladie est null
4,70,Male,NaN,87.7,168,La cellule type_maladie est null
...,...,...,...,...,...,...
995,19,Female,Obesity,105.1,190,Erreur transformation col taille_cm transfo 1
996,27,Male,Hypertension,118.2,162,Erreur transformation col taille_cm transfo 1
997,67,Male,Hypertension,109.1,162,Erreur transformation col taille_cm transfo 1
998,22,Female,Obesity,109.7,151,Erreur transformation col taille_cm transfo 1


--------------- Check Fin -----------


,age,sexe,type_maladie,poids_kg,taille_cm


,age,sexe,type_maladie,poids_kg,taille_cm,erreur
0,32,Male,NaN,58.1,164,La cellule type_maladie est null
1,78,Male,NaN,102.2,170,La cellule type_maladie est null
2,41,Male,NaN,83.7,154,La cellule type_maladie est null
3,39,Female,NaN,106.3,198,La cellule type_maladie est null
4,70,Male,NaN,87.7,168,La cellule type_maladie est null
...,...,...,...,...,...,...
995,19,Female,Obesity,105.1,190,Erreur transformation col taille_cm transfo 1
996,27,Male,Hypertension,118.2,162,Erreur transformation col taille_cm transfo 1
997,67,Male,Hypertension,109.1,162,Erreur transformation col taille_cm transfo 1
998,22,Female,Obesity,109.7,151,Erreur transformation col taille_cm transfo 1
